<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/DL-2026/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B4%D0%BB%D1%8F_%D1%81%D1%82%D1%83%D0%B4%D0%B5%D0%BD%D1%82%D0%BE%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 **Задание для студентов: Разработка интеллектуальной системы с мультиагентной архитектурой и RAG**

---

## 🎯 **Цель проекта**

Создать **интеллектуальную систему** на любую тему (на выбор студента): машинный перевод, генерация сказок, QA-система, анализ резюме, рекомендации, суммаризация, чат-бот, проверка кода и т.д. Главное – применить современный стек технологий и архитектурные паттерны.

**Система должна включать:**
- Мультиагентную архитектуру (минимум 3 агента + Supervisor)
- Retrieval-Augmented Generation (RAG) с гибридным поиском (BM25 + векторный поиск)
- Два варианта LLM: Ollama и HuggingFace Transformers
- REST API (FastAPI)
- Асинхронную обработку, кэширование, фоновые задачи (Celery)
- Human-in-the-Loop (обратная связь и обработка неоднозначных результатов)
- Контейнеризацию (Docker)
- Тестирование (pytest)
- Простой UI (Streamlit или аналог)

**Важно:** студенты **не** используют готовые фреймворки для агентов (LangChain, LangGraph и т.п.). Вся логика агентов и оркестрации реализуется самостоятельно. Готовый код не предоставляется, даётся только описание архитектуры и требований.

---

## 🏗️ **Примерная архитектура проекта (ориентир)**

Ниже приведена структура, которую можно взять за основу. Студенты вольны менять названия файлов и папок, но обязаны реализовать все перечисленные компоненты.

```
project/
├── backend/
│   ├── agents/            # Агенты и супервизор
│   │   ├── base.py        # Абстрактный класс Agent
│   │   ├── validator.py   # Агент проверки (rule-based + LLM)
│   │   ├── critic.py      # Агент оценки качества (LLM-as-Judge)
│   │   ├── editor.py      # Агент исправления/улучшения
│   │   └── supervisor.py  # Оркестратор цикла
│   ├── api/
│   │   ├── routes/        # Роуты FastAPI
│   │   ├── deps.py        # Зависимости (auth, DB)
│   │   └── main.py        # Точка входа приложения
│   ├── core/
│   │   ├── config.py      # Настройки (pydantic-settings)
│   │   ├── constants.py   # Константы
│   │   ├── models.py      # Pydantic-модели
│   │   └── exceptions.py  # Свои исключения
│   ├── data/
│   │   ├── knowledge_base.py    # Работа с базой знаний / справочниками
│   │   ├── examples_store.py    # Работа с коллекцией примеров
│   │   ├── cache.py       # Кэш
│   │   ├── embeddings.py  # Эмбеддинги (SentenceTransformer)
│   │   ├── llm_manager.py # Выбор между Ollama и HuggingFace
│   │   └── hf_model.py    # Обёртка для HF pipeline
│   ├── db/
│   │   ├── models.py      # SQLAlchemy-модели таблиц
│   │   ├── session.py     # Подключение к БД
│   │   └── database.py    # Экспорт
│   ├── hitl/              # Human-in-the-Loop
│   │   ├── active_learning.py # Очередь неоднозначных примеров
│   │   ├── feedback.py    # Обратная связь
│   │   └── updater.py     # Обновление базы знаний
│   ├── retrieval/
│   │   ├── bm25_index.py  # BM25
│   │   ├── vector_index.py# Векторный индекс (Qdrant)
│   │   ├── hybrid_retriever.py # Гибридный поиск + RRF
│   │   └── reranker.py    # (опционально) переранжирование
│   ├── utils/
│   │   ├── llm_client.py  # Обёртка для LLM
│   │   ├── llm_processing.py # Очистка ответов
│   │   ├── async_utils.py # Retry-декоратор
│   │   ├── logging.py     # Настройка логирования
│   │   ├── security.py    # JWT, пароли
│   │   └── text_processing.py # Токенизация, нормализация
│   └── worker/            # Celery
│       ├── celery_app.py  # Приложение Celery
│       └── tasks.py       # Задачи
├── frontend/              # ИЛИ streamlit_app/
│   ├── app.py             # Главная страница
│   ├── pages/             # Дополнительные страницы
│   └── utils/
│       ├── api_client.py  # Клиент для API
│       └── constants.py   # Справочники
├── scripts/
│   ├── create_admin.py    # Создание админа
│   ├── import_data.py     # Массовый импорт CSV/JSON
│   ├── run_eval.py        # Скрипт оценки качества
│   └── import/            # Скрипты добавления данных
├── tests/
│   ├── unit/              # Модульные тесты
│   └── integration/       # Интеграционные тесты
├── infrastructure/
│   ├── docker-compose.yml # Все сервисы
│   ├── Dockerfile         # Образ приложения
│   └── nginx.conf         # Прокси
├── data/
│   └── import/            # CSV/JSON-файлы с эталонными данными
├── pyproject.toml         # Poetry
└── run.py                 # Точка входа
```

---

## 🤖 **Требования к мультиагентной системе**

1. **Базовый класс агента**
   - Абстрактный класс `Agent` с методом `async def process(state) -> state`.
   - Хранит имя агента и последний результат.
   - Все агенты наследуются от него.

2. **Валидатор (Validator)**
   - Выполняет проверку результата по формальным правилам (минимальная длина, наличие обязательных полей, отсутствие повторов и т.п.).
   - Может вызывать LLM для дополнительной проверки.
   - Возвращает список ошибок (`errors`) и предупреждений (`warnings`), а также флаг `passed`.

3. **Критик (Critic)**
   - LLM-as-a-Judge: оценивает результат по нескольким критериям (например, релевантность, полнота, стиль, точность).
   - Использует few-shot prompting с примерами хороших и плохих ответов.
   - Возвращает баллы (0–10) по каждому критерию, общий балл и рекомендации.
   - В случае ошибки LLM (таймаут, неверный JSON) возвращает fallback-оценку (например, 5.0).

4. **Редактор (Editor)**
   - Принимает ошибки/предупреждения от Validator и рекомендации от Critic.
   - Исправляет или улучшает результат через LLM.
   - Возвращает исправленный результат и флаг, изменилось ли что-то.

5. **Супервизор (Supervisor)**
   - Оркеструет работу агентов в цикле:
     ```
     Validator -> Critic -> (если нужно) Editor -> Validator -> Critic -> ...
     ```
   - Ограничивает количество итераций (`max_iterations`).
   - Определяет критерий успеха (например, `validation_passed == True` и `critic_score >= 7`).
   - Ведёт лог всех шагов (`reasoning_log`).

---

## 🔍 **Требования к RAG**

1. **Гибридный поиск**
   - Реализовать два индекса: BM25 (лексический) и векторный (семантический, Qdrant).
   - Объединить результаты методом Reciprocal Rank Fusion (RRF).

2. **Эмбеддинги**
   - Использовать `sentence-transformers` (например, `paraphrase-multilingual-MiniLM-L12-v2`).
   - Оборачивать в класс-синглтон с ленивой загрузкой.

3. **Векторная БД**
   - Qdrant (или аналог: ChromaDB, FAISS) с коллекцией и cosine-метрикой.
   - Добавление, поиск с фильтрами по категориям/языкам (если применимо).

4. **Интеграция с LLM**
   - Контекст из найденных записей подаётся в промпт генеративной модели.

---

## 🧠 **Поддержка LLM: Ollama и HuggingFace**

1. **Ollama** — основной вариант для локального запуска.
   - HTTP API (`/api/generate`).
   - Поддержка разных моделей (например, qwen2.5, mistral, llama3.1).
   - Настройка `temperature` для разных агентов.

2. **HuggingFace Transformers** — запасной вариант для случаев, когда нужна конкретная модель или когда Ollama недоступен.
   - Асинхронная обёртка над `transformers.pipeline`.
   - Кэширование результатов.

3. **Выбор бэкенда** (через `LLMManager` или аналогичный класс)
   - Если задача поддерживается Ollama — использовать его.
   - Иначе переключаться на HuggingFace.

---

## 📊 **Сбор эталонных данных (500–1000+ записей)**

Студенты должны подготовить **обучающий набор** для базы знаний и справочников (аналог глоссария).

- **Справочник / база терминов**: не менее **500** уникальных записей (пары или структурированные объекты).
- **База знаний / коллекция примеров**: не менее **1000** записей (пары "вход-выход", "вопрос-ответ", "текст-резюме" и т.п.).
- Данные можно взять из открытых источников (например, датасеты с Kaggle, OPUS, Tatoeba, готовые корпуса) или сгенерировать/составить вручную.
- Формат CSV или JSON. Колонки зависят от предметной области, но должны быть согласованы с кодом импорта.

**Скрипт генерации**:
- Читает исходные файлы (JSON, CSV или txt).
- Сэмплирует нужное количество записей.
- При необходимости подсчитывает частотность и отбирает самые частотные для справочника.
- Сохраняет результат в CSV/JSON.

**Скрипт импорта** в БД:
- Читает CSV/JSON.
- Вставляет записи в таблицы через SQLAlchemy, избегая дубликатов.

---

## 🛠️ **Backend требования**

1. **FastAPI** со следующими группами эндпоинтов (названия адаптируются под задачу):
   - Основной запрос (например, `/generate`, `/answer`, `/analyze`).
   - Мульти-запрос (несколько вариантов/языков/категорий).
   - Пакетная обработка.
   - Ансамбль моделей с выбором лучшего варианта.
   - Административные эндпоинты для управления справочником и базой знаний.
   - Аутентификация: регистрация, вход, получение информации.
   - Оценка качества (BLEU, chrF, BERTScore, METEOR для текстовых задач или свои метрики).
   - Human-in-the-Loop: получение неоднозначных примеров, подтверждение/отклонение.
   - Проверка состояния сервисов (`/health`).

2. **Аутентификация**:
   - JWT токены.
   - Роли: `user`, `editor`, `admin`.
   - Хэширование паролей (bcrypt).

3. **База данных**:
   - SQLAlchemy (async) с поддержкой PostgreSQL (основной) и SQLite (для тестов).
   - Таблицы: пользователи, справочник, база знаний, неоднозначные примеры, кэш.

4. **Кэширование**:
   - Таблица в БД или Redis.
   - Ключ: например, `(input_text, category, parameters)`.
   - Увеличивать счётчик попаданий.

5. **Фоновые задачи (Celery)**:
   - Переиндексация базы знаний.
   - Пакетная обработка.
   - Очистка устаревшего кэша.
   - Массовое добавление записей.

6. **Логирование**:
   - Структурированные логи (JSON optional).
   - Ротация файлов.
   - Разные уровни для разных модулей.

---

## 🧪 **Тестирование**

- Использовать **pytest** и **pytest-asyncio**.
- Покрытие тестами не менее **70%** (можно проверить `pytest-cov`).
- Обязательные категории:
  - Юнит-тесты для каждого агента (проверка логики, обработка ошибок).
  - Юнит-тесты для retrieval (BM25, vector, RRF).
  - Юнит-тесты для API-роутов (с mock-объектами).
  - Интеграционные тесты (через `TestClient`).
- Использовать `monkeypatch` и `unittest.mock` для изоляции внешних сервисов (Ollama, Qdrant, Redis).
- Создавать фикстуры для тестовой БД и тестовых данных.

---

## 🐳 **Инфраструктура**

1. **Poetry** для управления зависимостями (`pyproject.toml`).
2. **Docker Compose** со следующими сервисами:
   - `db` — PostgreSQL 16
   - `redis` — Redis 7
   - `qdrant` — векторная БД
   - `ollama` — LLM (с инициализацией и подгрузкой моделей)
   - `app` — FastAPI
   - `worker` — Celery worker
   - `beat` — Celery beat
   - `nginx` — прокси
3. **Healthcheck** для каждого сервиса.
4. **Nginx** конфигурация с проксированием, gzip, security headers.
5. **Dockerfile** для сборки образа приложения.

---

## 🖥️ **Frontend (Streamlit или аналог)**

- Страница входа/регистрации.
- Основная страница с формой для выполнения задачи (генерация/ответ/анализ).
- Страница пакетной обработки (загрузка файла, массовая обработка).
- Страница управления справочником и базой знаний (CRUD, импорт CSV).
- Страница Human-in-the-Loop (просмотр неоднозначных примеров, подтверждение/отклонение).
- Страница оценки качества (запуск метрик).
- Страница мониторинга (статусы сервисов, статистика кэша).

---

## 📋 **Критерии оценки**

| Критерий | Вес |
|----------|-----|
| Архитектура и чистота кода | 30% |
| Функциональность (работающие агенты, RAG, API) | 30% |
| Тестирование (покрытие, качество) | 20% |
| Документация (README, инструкции) | 10% |
| Инфраструктура (Docker, Poetry) | 10% |

**Бонусы** (до +10%): мониторинг, CI/CD, WebSocket, rate limiting, A/B тесты.

---

## 📦 **Итоговый результат**

Студент должен предоставить:

1. Репозиторий с полным кодом.
2. README с инструкциями по запуску (через Docker одной командой).
3. Набор эталонных данных (CSV/JSON) объёмом от 500 до 1000+ записей.
4. Скрипты для генерации и импорта данных.
5. Скрипт `run_eval.py` для оценки качества и сравнения режимов (baseline, +справочник, +RAG, +agents).
6. Отчёт с описанием архитектуры и принятых решений.

---

**Удачи!** Помните: главное — не просто скопировать идею, а понять принципы и реализовать свою систему.